# NB01 — Export and Reshape

**Input:** `data/raw/database.db` — SQLite database van de surveyapplicatie  
**Output:** `data/processed/comparisons.csv` — één rij per (respondent x taak) vergelijking  

**Doel:** Verbind de vier relevante tabellen (Response, Task_set, Act1_task, Timestamp) tot een
lange tabel van paarsgewijze vergelijkingen. Elke rij legt vast welk kruispunt als alternatief 1
werd getoond, welk als alternatief 2, wat de respondent koos, en welk kruispunt daarmee won
en verloor.

**Kolommen in de output:**

| Kolom | Inhoud |
|---|---|
| `respondent_id` | Uniek respondent-ID uit `Response` |
| `task_position` | Taakpositie binnen de set (1-36) |
| `task_id` | Werkelijk taak-ID uit `Task_set` |
| `intersection_a` | Kruispunt-ID van alternatief 1 (uit bestandsnaam) |
| `intersection_b` | Kruispunt-ID van alternatief 2 (uit bestandsnaam) |
| `choice` | Ruwe keuze: `'1'` of `'2'` |
| `winner` | Kruispunt-ID van de gekozen (gevaarlijker geachte) optie |
| `loser` | Kruispunt-ID van de verworpen optie |

**Compleetheid-filter:** Respondenten met `act1_t36 IS NOT NULL` worden als volledig beschouwd.
Pas `MIN_TASKS_COMPLETED` aan om gedeeltelijk-afgeronde respondenten toch mee te nemen.

## Stap 1 — Paden en parameters

Definieer het pad naar de database en de outputmap. `MIN_TASKS_COMPLETED` bepaalt hoeveel taken
een respondent minimaal beantwoord moet hebben om meegenomen te worden in de analyse.

In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

# Notebooks staan in notebooks/, data staat in de bovenliggende map bradley_terry/
BASE_DIR = Path("..")
DB_PATH  = BASE_DIR / "data" / "raw" / "database.db"
OUT_DIR  = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_TASKS             = 36   # alleen t1..t36 zijn relevant; t37..t300 zijn ongebruikte framework-kolommen
MIN_TASKS_COMPLETED = 36   # verlaag naar bijv. 30 om deels-afgeronde respondenten te includeren

print(f"Database: {DB_PATH.resolve()}")
print(f"Minimum taken beantwoord om respondent mee te nemen: {MIN_TASKS_COMPLETED}")

Database: C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\bradley_terry\data\raw\database.db
Minimum taken beantwoord om respondent mee te nemen: 36


## Stap 2 — Tabellen laden uit SQLite

Laad de drie benodigde tabellen uit de database:
- **Response** — een rij per respondent, met hun keuzes per taakpositie (`act1_t1`..`act1_t36`)
- **Task_set** — welke taak-ID op welke positie stond per set
- **Act1_task** — de twee fotopaden (en dus kruispunt-ID's) per taak

We selecteren alleen de relevante kolommen om het geheugengebruik laag te houden.

In [6]:
con = sqlite3.connect(DB_PATH)

# Response: een rij per respondent, kolommen act1_t1..act1_t300 (we gebruiken t1..t36)
response_cols = ["respondent_id", "set_id"] + [f"act1_t{i}" for i in range(1, N_TASKS + 1)]
response_df   = pd.read_sql(f"SELECT {', '.join(response_cols)} FROM Response", con)

# Task_set: koppelt set_id aan de 36 taak-ID's op elke positie
taskset_cols = ["set_id"] + [f"act1_task_{i}" for i in range(1, N_TASKS + 1)]
taskset_df   = pd.read_sql(f"SELECT {', '.join(taskset_cols)} FROM Task_set", con)

# Act1_task: geeft de twee fotopaden per taak
task_df = pd.read_sql(
    "SELECT task_id, alt1_att1_streetview, alt2_att1_streetview FROM Act1_task",
    con
)

con.close()

print(f"Respondenten geladen:  {len(response_df)}")
print(f"Task sets geladen:     {len(taskset_df)}")
print(f"Taken geladen:         {len(task_df)}")
response_df.head(2)

Respondenten geladen:  76
Task sets geladen:     14
Taken geladen:         504


,respondent_id,set_id,act1_t1,act1_t2,act1_t3,act1_t4,act1_t5,act1_t6,act1_t7,act1_t8,...,act1_t27,act1_t28,act1_t29,act1_t30,act1_t31,act1_t32,act1_t33,act1_t34,act1_t35,act1_t36
0,490974580,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,504992548,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Stap 3 — Compleetheid-filter

Tel voor elke respondent hoeveel taken ze beantwoord hebben (waarde `'1'` of `'2'`, niet null).
Houd alleen respondenten over die aan het minimumdrempel voldoen.

De `finished`-kolom ontbreekt in de database (bekende bug in het survey-framework), vandaar dat
we dit zelf berekenen.

In [7]:
task_answer_cols = [f"act1_t{i}" for i in range(1, N_TASKS + 1)]

# Tel het aantal beantwoorde taken per respondent
response_df["n_answered"] = response_df[task_answer_cols].isin(["1", "2"]).sum(axis=1)

complete_df = response_df[response_df["n_answered"] >= MIN_TASKS_COMPLETED].copy()

print(f"Totaal respondenten:      {len(response_df)}")
print(f"Volledige respondenten:   {len(complete_df)} (>= {MIN_TASKS_COMPLETED} taken beantwoord)")
complete_df[["respondent_id", "set_id", "n_answered"]].head()

Totaal respondenten:      76
Volledige respondenten:   6 (>= 36 taken beantwoord)


,respondent_id,set_id,n_answered
4,565642399,1,36
17,851656988,1,36
27,935789013,1,36
28,940909515,1,36
37,1069040705,2,36


## Stap 4 — Kruispunt-ID's extraheren uit fotopaden

Het kruispunt-ID zit verborgen in de bestandsnaam van de streetview-foto.
Voorbeeld: `/assets/images/182270019.jpeg` -> `182270019`.

We verwerken dit vooraf voor alle taken in een keer, zodat we de extractie niet steeds opnieuw
hoeven te doen in de binnenste lus. Het resultaat wordt opgeslagen als een dictionary voor
snelle O(1)-opzoeking op `task_id`.

In [8]:
def path_to_intersection_id(photo_path: str) -> str:
    # Haal de bestandsnaam op zonder map en zonder extensie
    return Path(photo_path).stem

task_df["intersection_a"] = task_df["alt1_att1_streetview"].apply(path_to_intersection_id)
task_df["intersection_b"] = task_df["alt2_att1_streetview"].apply(path_to_intersection_id)

# Sla op als dict voor snelle opzoeking: task_id -> {intersection_a, intersection_b}
task_lookup = task_df.set_index("task_id")[["intersection_a", "intersection_b"]].to_dict("index")

print("Voorbeeld task-opzoeking:")
sample_id = list(task_lookup.keys())[0]
print(f"  task_id={sample_id} -> {task_lookup[sample_id]}")

Voorbeeld task-opzoeking:
  task_id=1 -> {'intersection_a': '182270019', 'intersection_b': '186275023'}


## Stap 5 — Task-set opzoektabel bouwen

Elke respondent kreeg een van de 14 task sets. Per set leggen we vast welk taak-ID op welke
positie (1..36) stond. Dit geeft ons een geneste dictionary: `set_id -> positie -> task_id`.

We doen dit vooraf zodat we in de hoofdlus per respondent direct de taak-ID's kunnen opzoeken
zonder de DataFrame steeds opnieuw te filteren.

In [9]:
taskset_df = taskset_df.set_index("set_id")

# Bouw: {set_id: {positie: task_id, ...}, ...}
set_task_lookup = {
    set_id: {
        pos: row[f"act1_task_{pos}"]
        for pos in range(1, N_TASKS + 1)
    }
    for set_id, row in taskset_df.iterrows()
}

print(f"Sets geindexeerd: {list(set_task_lookup.keys())}")

Sets geindexeerd: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


## Stap 6 — Hoofdjoin: respondent x taakpositie -> vergelijking

Dit is de kernstap: voor elke respondent en elke taakpositie combineren we:
1. De keuze van de respondent (`'1'` of `'2'`)
2. Het taak-ID op die positie (via de task-set opzoektabel)
3. De twee kruispunt-ID's van die taak (via de task-opzoektabel)

Op basis van de keuze bepalen we wie `winner` en wie `loser` is.
Niet-beantwoorde posities (NULL) worden overgeslagen.

In [10]:
records = []

for _, respondent in complete_df.iterrows():
    r_id   = respondent["respondent_id"]
    set_id = respondent["set_id"]

    for pos in range(1, N_TASKS + 1):
        # Ruwe keuze: '1', '2', of NaN als niet beantwoord
        choice = respondent[f"act1_t{pos}"]
        if choice not in ("1", "2"):  # sla onbeantwoorde taken over
            continue

        # Zoek het taak-ID op voor deze positie in de set van de respondent
        task_id = set_task_lookup[set_id][pos]

        # Zoek de kruispunt-ID's op voor deze taak
        inter = task_lookup[task_id]
        int_a = inter["intersection_a"]
        int_b = inter["intersection_b"]

        # Keuze '1' = alternatief 1 gevaarlijker; keuze '2' = alternatief 2 gevaarlijker
        if choice == "1":
            winner, loser = int_a, int_b
        else:
            winner, loser = int_b, int_a

        records.append({
            "respondent_id":  r_id,
            "task_position":  pos,
            "task_id":        task_id,
            "intersection_a": int_a,
            "intersection_b": int_b,
            "choice":         choice,
            "winner":         winner,
            "loser":          loser,
        })

comparisons_df = pd.DataFrame(records)

print(f"Totaal vergelijkingen:       {len(comparisons_df)}")
print(f"Unieke respondenten:         {comparisons_df['respondent_id'].nunique()}")
print(f"Unieke taken:                {comparisons_df['task_id'].nunique()}")
print(f"Unieke kruispunten:          {pd.concat([comparisons_df['intersection_a'], comparisons_df['intersection_b']]).nunique()}")
comparisons_df.head(5)

Totaal vergelijkingen:       216
Unieke respondenten:         6
Unieke taken:                72
Unieke kruispunten:          68


,respondent_id,task_position,task_id,intersection_a,intersection_b,choice,winner,loser
0,565642399,1,1,182270019,186275023,2,186275023,182270019
1,565642399,2,2,184273072,600111024,2,600111024,184273072
2,565642399,3,3,181265038,182272213,2,182272213,181265038
3,565642399,4,4,183272020,188278029,2,188278029,183272020
4,565642399,5,5,181277036,188266127,2,188266127,181277036


## Stap 7 — Sanity checks

Twee basiscontroles om te bevestigen dat de join correct is verlopen:
1. Geen enkele rij mag hetzelfde kruispunt aan beide kanten hebben.
2. Bij keuze `'1'` moet de winner gelijk zijn aan `intersection_a`; bij `'2'` aan `intersection_b`.

Een falende assert hier wijst op een fout in de join-logica of de data.

In [11]:
# Controleer: geen zelf-vergelijkingen (kruispunt vs. zichzelf)
self_comparisons = comparisons_df[comparisons_df["intersection_a"] == comparisons_df["intersection_b"]]
assert len(self_comparisons) == 0, f"{len(self_comparisons)} zelf-vergelijkingen gevonden!"

# Controleer: winner-toewijzing klopt met de keuze
choice_1_correct = (comparisons_df[comparisons_df["choice"] == "1"]["winner"] ==
                    comparisons_df[comparisons_df["choice"] == "1"]["intersection_a"]).all()
choice_2_correct = (comparisons_df[comparisons_df["choice"] == "2"]["winner"] ==
                    comparisons_df[comparisons_df["choice"] == "2"]["intersection_b"]).all()
assert choice_1_correct and choice_2_correct, "Winner-toewijzing klopt niet!"

print("Alle sanity checks geslaagd.")
print(f"\nVergelijkingen per respondent:")
print(comparisons_df.groupby("respondent_id").size().describe())

Alle sanity checks geslaagd.

Vergelijkingen per respondent:
count     6.0
mean     36.0
std       0.0
min      36.0
25%      36.0
50%      36.0
75%      36.0
max      36.0
dtype: float64


## Stap 8 — Opslaan

Sla de lange vergelijkingstabel op als CSV. Dit bestand is de input voor NB02 (wins-matrix)
en NB04 (validatie).

In [12]:
out_path = OUT_DIR / "comparisons.csv"
comparisons_df.to_csv(out_path, index=False)
print(f"Opgeslagen: {len(comparisons_df)} rijen -> {out_path.resolve()}")

Opgeslagen: 216 rijen -> C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\bradley_terry\data\processed\comparisons.csv
